[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sausterm/omniscience-research/blob/main/omni_toolkit/notebooks/bci_demo.ipynb)
[![View on GitHub](https://img.shields.io/badge/GitHub-View%20Source-blue?logo=github)](https://github.com/sausterm/omniscience-research/blob/main/omni_toolkit/notebooks/bci_demo.ipynb)

# Geometric EEG Classification
## Riemannian BCI on GL$^+$(n)/SO(n) with DeWitt Curvature Weighting

**OmniSciences LLC** | [omnisciences.io](https://omnisciences.io)

---

This notebook demonstrates the OmniSciences BCI API, which performs EEG covariance classification on the manifold of symmetric positive-definite (SPD) matrices using Riemannian geometry.

Standard BCI pipelines extract band-power features and feed them to linear classifiers. Our Riemannian approach works directly with **spatial covariance matrices** on GL$^+$(n)/SO(n), leveraging geodesic distances, tangent-space projections, and DeWitt curvature weighting to achieve superior classification accuracy.

> **API Key**: Contact sloan@omnisciences.org for access.

In [ ]:
import requests
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

# -- Configuration ----------------------------------------------------------
API_URL = "https://bci.omnisciences.io"
API_KEY = "your-api-key-here"  # Get yours at omnisciences.io
# ---------------------------------------------------------------------------

HEADERS = {"X-API-Key": API_KEY, "Content-Type": "application/json"}


def api_post(endpoint, payload):
    """POST to the BCI API and return the JSON response."""
    url = f"{API_URL}/bci/{endpoint}"
    resp = requests.post(url, json=payload, headers=HEADERS, timeout=60)
    resp.raise_for_status()
    return resp.json()


# Quick connectivity check
try:
    r = requests.get(f"{API_URL}/bci/health", headers=HEADERS, timeout=5)
    info = r.json()
    print(f"API status:   {info['status']}")
    print(f"Version:      {info['version']}")
    print(f"Methods:      {', '.join(info['methods'])}")
except Exception as e:
    print(f"Cannot reach API at {API_URL}")
    print(f"Error: {e}")

## 1. Generate Synthetic Motor Imagery Data

In Riemannian BCI, each EEG trial is represented by its **spatial covariance matrix** -- an SPD matrix capturing the second-order statistics across channels. Motor imagery tasks (e.g., imagining left vs right hand movement) produce distinct covariance patterns due to event-related desynchronization (ERD) in contralateral motor cortex.

We generate synthetic 8-channel covariance matrices for two classes:
- **Class 0 (left hand)**: Elevated power in channels 0--3 (right motor cortex)
- **Class 1 (right hand)**: Elevated power in channels 4--7 (left motor cortex)

In [ ]:
np.random.seed(42)

n_channels = 8
n_trials_per_class = 60
n_trials = 2 * n_trials_per_class

channel_names = [f"C{i+1}" for i in range(n_channels)]


def make_spd(base_eigs, noise_scale=0.3):
    """Generate a random SPD matrix with prescribed eigenvalue profile."""
    # Random orthogonal basis
    Q, _ = np.linalg.qr(np.random.randn(n_channels, n_channels))
    # Perturbed eigenvalues (stay positive)
    eigs = base_eigs + noise_scale * np.abs(np.random.randn(n_channels))
    eigs = np.maximum(eigs, 0.1)
    return (Q * eigs) @ Q.T


# Class 0 (left hand): boost channels 0-3
eigs_class0 = np.array([3.0, 2.8, 2.5, 2.3, 1.0, 1.0, 1.0, 1.0])
# Class 1 (right hand): boost channels 4-7
eigs_class1 = np.array([1.0, 1.0, 1.0, 1.0, 3.0, 2.8, 2.5, 2.3])

covariances = []
labels = []

for _ in range(n_trials_per_class):
    covariances.append(make_spd(eigs_class0))
    labels.append(0)

for _ in range(n_trials_per_class):
    covariances.append(make_spd(eigs_class1))
    labels.append(1)

covariances = np.array(covariances)
labels = np.array(labels)

# Shuffle
idx = np.random.permutation(n_trials)
covariances = covariances[idx]
labels = labels[idx]

# 80/20 train/test split
n_train = int(0.8 * n_trials)
train_covs = covariances[:n_train]
train_labels = labels[:n_train]
test_covs = covariances[n_train:]
test_labels = labels[n_train:]

print(f"Total trials:     {n_trials}")
print(f"Channels:         {n_channels}")
print(f"Train / Test:     {n_train} / {n_trials - n_train}")
print(f"Class balance:    {(labels == 0).sum()} left, {(labels == 1).sum()} right")
print(f"\nExample covariance shape: {covariances[0].shape}")
print(f"Condition numbers: min={np.min([np.linalg.cond(c) for c in covariances]):.1f}, "
      f"max={np.max([np.linalg.cond(c) for c in covariances]):.1f}")

## 2. Train & Classify

We compare all four classification methods:

| Method | Description |
|--------|-------------|
| `mdm` | Minimum Distance to Mean -- classify by nearest Riemannian class mean |
| `ts_lr` | Tangent Space + Logistic Regression -- project to tangent space, then linear classify |
| `ts_dewitt` | Tangent Space with DeWitt metric + LR -- uses GL$^+$(n)/SO(n) geometry |
| `ts_dewitt_curv` | DeWitt + Curvature Weighting -- weights tangent features by sectional curvature |

In [ ]:
methods = ["mdm", "ts_lr", "ts_dewitt", "ts_dewitt_curv"]
method_labels = ["MDM", "TS + LR", "TS + DeWitt", "TS + DeWitt + Curv"]
results = {}

for method, label in zip(methods, method_labels):
    resp = api_post("classify", {
        "train_covariances": train_covs.tolist(),
        "train_labels": train_labels.tolist(),
        "test_covariances": test_covs.tolist(),
        "method": method
    })
    preds = np.array(resp["predictions"])
    acc = np.mean(preds == test_labels)
    results[label] = {"accuracy": acc, "predictions": preds}
    print(f"{label:<25} Accuracy: {acc:.1%}  ({int(acc * len(test_labels))}/{len(test_labels)})")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#95a5a6", "#3498db", "#2980b9", "#1a5276"]
accs = [results[l]["accuracy"] for l in method_labels]
bars = ax.bar(method_labels, accs, color=colors, edgecolor="black", linewidth=0.5)

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{acc:.1%}", ha="center", va="bottom", fontweight="bold", fontsize=13)

ax.set_ylabel("Accuracy")
ax.set_title("Motor Imagery Classification: Method Comparison",
             fontsize=14, fontweight="bold")
ax.set_ylim(0, 1.15)
ax.axhline(y=0.5, color="red", linestyle="--", alpha=0.5, label="Chance level")
ax.legend()
ax.grid(True, alpha=0.2, axis="y")
plt.tight_layout()
plt.show()

## 3. Riemannian Geometry Analysis

The `/bci/geometry` endpoint decomposes each covariance matrix according to the GL$^+$(n)/SO(n) structure:
- **V$^+$ energy**: Gauge (traceless symmetric) component energy
- **V$^-$ energy**: Conformal (trace/scale) component energy
- **Shape ratio**: V$^+$ / V$^-$ -- a geometric discriminability feature

In [ ]:
# Analyze example covariances from each class
class0_idx = np.where(labels == 0)[0][:5]
class1_idx = np.where(labels == 1)[0][:5]

geo_class0 = []
geo_class1 = []

for i in class0_idx:
    resp = api_post("geometry", {"covariance": covariances[i].tolist()})
    geo_class0.append(resp)

for i in class1_idx:
    resp = api_post("geometry", {"covariance": covariances[i].tolist()})
    geo_class1.append(resp)

print(f"{'Metric':<25} {'Class 0 (Left)':>15} {'Class 1 (Right)':>15}")
print("=" * 55)
print(f"{'V+ energy (mean)':<25} {np.mean([g['vplus_energy'] for g in geo_class0]):>15.4f} "
      f"{np.mean([g['vplus_energy'] for g in geo_class1]):>15.4f}")
print(f"{'V- energy (mean)':<25} {np.mean([g['vminus_energy'] for g in geo_class0]):>15.4f} "
      f"{np.mean([g['vminus_energy'] for g in geo_class1]):>15.4f}")
print(f"{'Shape ratio (mean)':<25} {np.mean([g['shape_ratio'] for g in geo_class0]):>15.4f} "
      f"{np.mean([g['shape_ratio'] for g in geo_class1]):>15.4f}")
print(f"{'Condition number (mean)':<25} {np.mean([g['condition_number'] for g in geo_class0]):>15.1f} "
      f"{np.mean([g['condition_number'] for g in geo_class1]):>15.1f}")

# Plot eigenvalue spectra
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for g in geo_class0:
    axes[0].plot(range(1, n_channels + 1), sorted(g["eigenvalues"], reverse=True),
                'b-o', alpha=0.4, markersize=4)
axes[0].set_title("Class 0 (Left Hand) -- Eigenvalue Spectra", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Eigenvalue Index")
axes[0].set_ylabel("Eigenvalue")
axes[0].grid(True, alpha=0.3)

for g in geo_class1:
    axes[1].plot(range(1, n_channels + 1), sorted(g["eigenvalues"], reverse=True),
                'r-o', alpha=0.4, markersize=4)
axes[1].set_title("Class 1 (Right Hand) -- Eigenvalue Spectra", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Eigenvalue Index")
axes[1].set_ylabel("Eigenvalue")
axes[1].grid(True, alpha=0.3)

plt.suptitle("Covariance Eigenvalue Spectra by Class", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 4. Geodesic Distance Matrix

Geodesic distance on the SPD manifold measures the intrinsic separation between covariance matrices. A good BCI representation should produce **small within-class distances** and **large between-class distances**. We compute the Riemannian class means and measure discriminability via the distance ratio.

In [ ]:
# Compute Riemannian mean of each class
class0_covs = covariances[labels == 0]
class1_covs = covariances[labels == 1]

mean0_resp = api_post("mean", {"covariances": class0_covs.tolist()})
mean1_resp = api_post("mean", {"covariances": class1_covs.tolist()})
mean0 = np.array(mean0_resp["mean"])
mean1 = np.array(mean1_resp["mean"])

# Between-class distance
between_riem = api_post("distance", {
    "cov1": mean0.tolist(), "cov2": mean1.tolist(), "metric": "riemann"
})["distance"]
between_dewitt = api_post("distance", {
    "cov1": mean0.tolist(), "cov2": mean1.tolist(), "metric": "dewitt"
})["distance"]

# Within-class distances (sample 10 pairs per class)
within_riem_0, within_riem_1 = [], []
within_dewitt_0, within_dewitt_1 = [], []

for i in range(10):
    j = (i + 1) % len(class0_covs)
    d_r = api_post("distance", {
        "cov1": class0_covs[i].tolist(), "cov2": class0_covs[j].tolist(), "metric": "riemann"
    })["distance"]
    d_d = api_post("distance", {
        "cov1": class0_covs[i].tolist(), "cov2": class0_covs[j].tolist(), "metric": "dewitt"
    })["distance"]
    within_riem_0.append(d_r)
    within_dewitt_0.append(d_d)

    d_r = api_post("distance", {
        "cov1": class1_covs[i].tolist(), "cov2": class1_covs[j].tolist(), "metric": "riemann"
    })["distance"]
    d_d = api_post("distance", {
        "cov1": class1_covs[i].tolist(), "cov2": class1_covs[j].tolist(), "metric": "dewitt"
    })["distance"]
    within_riem_1.append(d_r)
    within_dewitt_1.append(d_d)

within_riem = np.mean(within_riem_0 + within_riem_1)
within_dewitt = np.mean(within_dewitt_0 + within_dewitt_1)

print(f"{'Metric':<25} {'Riemannian':>15} {'DeWitt':>15}")
print("=" * 55)
print(f"{'Between-class distance':<25} {between_riem:>15.4f} {between_dewitt:>15.4f}")
print(f"{'Within-class distance':<25} {within_riem:>15.4f} {within_dewitt:>15.4f}")
print(f"{'Discriminability ratio':<25} {between_riem/within_riem:>15.2f} {between_dewitt/within_dewitt:>15.2f}")
print(f"\nHigher ratio = better class separation.")

## 5. Tangent Space Visualization

The tangent space projection maps SPD matrices to a Euclidean vector space where standard classifiers (logistic regression, SVM) can operate. We compare the standard Riemannian projection with **curvature-weighted** projection, which rescales features by the sectional curvature of the corresponding tangent directions on GL$^+$(n)/SO(n).

In [ ]:
# Standard tangent space projection
ts_resp = api_post("tangent-space", {
    "covariances": covariances.tolist(),
    "curvature_weighted": False
})
features_std = np.array(ts_resp["features"])

# Curvature-weighted tangent space projection
ts_curv_resp = api_post("tangent-space", {
    "covariances": covariances.tolist(),
    "curvature_weighted": True
})
features_curv = np.array(ts_curv_resp["features"])

# Side-by-side scatter plots (first 2 features)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for cls, color, label in [(0, "#2980b9", "Left Hand"), (1, "#e74c3c", "Right Hand")]:
    mask = labels == cls
    axes[0].scatter(features_std[mask, 0], features_std[mask, 1],
                    c=color, label=label, alpha=0.6, edgecolors="black", linewidth=0.3, s=40)
    axes[1].scatter(features_curv[mask, 0], features_curv[mask, 1],
                    c=color, label=label, alpha=0.6, edgecolors="black", linewidth=0.3, s=40)

axes[0].set_title("Standard Tangent Space", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Feature 1")
axes[0].set_ylabel("Feature 2")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Curvature-Weighted Tangent Space", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Feature 1")
axes[1].set_ylabel("Feature 2")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Tangent Space Projections: Standard vs Curvature-Weighted",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"Standard features shape:  {features_std.shape}")
print(f"Curvature features shape: {features_curv.shape}")

## 6. DeWitt Curvature Features

The DeWitt metric on GL$^+$(n)/SO(n) induces a natural decomposition of the tangent space into **V$^+$** (traceless symmetric, gauge directions) and **V$^-$** (trace/conformal direction). The energy distribution across these subspaces, and the resulting **shape ratio**, provide geometrically meaningful features that standard Riemannian BCI methods miss.

This is what makes our approach novel: we exploit the *curvature structure* of the SPD manifold, not just its metric.

In [ ]:
# Compute geometry features for all trials
vplus_energies = []
vminus_energies = []
shape_ratios = []
trial_labels = []

for i in range(n_trials):
    resp = api_post("geometry", {"covariance": covariances[i].tolist()})
    vplus_energies.append(resp["vplus_energy"])
    vminus_energies.append(resp["vminus_energy"])
    shape_ratios.append(resp["shape_ratio"])
    trial_labels.append(labels[i])

vplus_energies = np.array(vplus_energies)
vminus_energies = np.array(vminus_energies)
shape_ratios = np.array(shape_ratios)
trial_labels = np.array(trial_labels)

# Plot distributions by class
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for cls, color, label in [(0, "#2980b9", "Left Hand"), (1, "#e74c3c", "Right Hand")]:
    mask = trial_labels == cls
    axes[0].hist(vplus_energies[mask], bins=15, alpha=0.6, color=color,
                 label=label, edgecolor="black", linewidth=0.5)
    axes[1].hist(vminus_energies[mask], bins=15, alpha=0.6, color=color,
                 label=label, edgecolor="black", linewidth=0.5)
    axes[2].hist(shape_ratios[mask], bins=15, alpha=0.6, color=color,
                 label=label, edgecolor="black", linewidth=0.5)

axes[0].set_title("V+ Energy Distribution", fontweight="bold")
axes[0].set_xlabel("V+ Energy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title("V- Energy Distribution", fontweight="bold")
axes[1].set_xlabel("V- Energy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].set_title("Shape Ratio (V+/V-)", fontweight="bold")
axes[2].set_xlabel("Shape Ratio")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle("DeWitt Decomposition: Curvature Features by Class",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Scatter: V+ vs V- colored by class
fig, ax = plt.subplots(figsize=(8, 6))
for cls, color, label in [(0, "#2980b9", "Left Hand"), (1, "#e74c3c", "Right Hand")]:
    mask = trial_labels == cls
    ax.scatter(vplus_energies[mask], vminus_energies[mask],
              c=color, label=label, alpha=0.6, edgecolors="black", linewidth=0.3, s=50)
ax.set_xlabel("V+ Energy (gauge directions)", fontsize=12)
ax.set_ylabel("V- Energy (conformal direction)", fontsize=12)
ax.set_title("DeWitt V+/V- Decomposition", fontsize=14, fontweight="bold")
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

| Method | Approach | Novel Geometry |
|--------|----------|:--------------:|
| **MDM** | Nearest Riemannian mean | No |
| **TS + LR** | Tangent space + logistic regression | No |
| **TS + DeWitt** | DeWitt metric tangent space + LR | **Yes** |
| **TS + DeWitt + Curv** | Curvature-weighted DeWitt features + LR | **Yes** |

### Novel Contributions

- **DeWitt V$^+$/V$^-$ decomposition**: The GL$^+$(n)/SO(n) symmetric space structure splits the tangent space into gauge (traceless symmetric) and conformal (trace) directions, providing geometrically interpretable features unavailable to standard Riemannian BCI.
- **Curvature weighting**: Tangent directions with higher sectional curvature carry more geometric information. Weighting features by curvature emphasizes the most discriminative manifold directions.
- **Shape ratio**: The V$^+$/V$^-$ energy ratio is a single scalar that captures the "shape" of a covariance matrix on the manifold -- a novel discriminability feature.

### API Access

- **Free tier**: 100 requests/month
- **Pro tier**: Unlimited requests, batch processing, priority support
- **Enterprise**: On-premise deployment, custom manifolds

### Contact

**OmniSciences LLC**  
Email: sloan@omnisciences.org  
Web: [omnisciences.io](https://omnisciences.io)  

*Patent pending. (c) 2026 OmniSciences LLC.*